#### Big data analytics bonus
Name : Nadine hassan
ID : 231005869

In [8]:
import os, re, zlib
from concurrent.futures import ThreadPoolExecutor

NUM_REDUCERS = 2
IN_DIR = "notes"
OUT_DIR = "out"

create sample input

In [2]:
os.makedirs(IN_DIR, exist_ok=True)
samples = {
    "note1.txt": "car damage fire roof",
    "note2.txt": "theft car water damage",
    "note3.txt": "fire roof water car",
}
for name, text in samples.items():
    with open(os.path.join(IN_DIR, name), "w") as f:
        f.write(text)
print(os.listdir(IN_DIR))

['note1.txt', 'note2.txt', 'note3.txt']


In [3]:
def map_phase(path):
    pairs = []
    with open(path) as f:
        for word in re.findall(r"[a-z']+", f.read().lower()):
            pairs.append((word, 1))
    return pairs

print(map_phase("notes/note1.txt"))  # quick test

[('car', 1), ('damage', 1), ('fire', 1), ('roof', 1)]


In [4]:
def combine_and_partition(pairs):
    # Combine: local subtotals
    local = {}
    for word, one in pairs:
        local[word] = local.get(word, 0) + one

    # Partition: same word always goes to the same reducer
    buckets = [[] for _ in range(NUM_REDUCERS)]
    for word, count in local.items():
        r = zlib.crc32(word.encode()) % NUM_REDUCERS
        buckets[r].append((word, count))
    return buckets

def mapper(path):
    return combine_and_partition(map_phase(path))

In [5]:
splits = [os.path.join(IN_DIR, n) for n in sorted(os.listdir(IN_DIR)) if n.endswith(".txt")]

with ThreadPoolExecutor(max_workers=len(splits)) as pool:
    mapper_outputs = list(pool.map(mapper, splits))

for i, out in enumerate(mapper_outputs):
    print(f"mapper {i}:", out)

mapper 0: [[('damage', 1)], [('car', 1), ('fire', 1), ('roof', 1)]]
mapper 1: [[('water', 1), ('damage', 1)], [('theft', 1), ('car', 1)]]
mapper 2: [[('water', 1)], [('fire', 1), ('roof', 1), ('car', 1)]]


In [6]:
def reduce_phase(pairs):
    # Shuffle: group values by word
    grouped = {}
    for word, count in pairs:
        grouped.setdefault(word, []).append(count)
    # Sort by word, then Reduce: sum
    return [(w, sum(grouped[w])) for w in sorted(grouped)]

In [7]:
os.makedirs(OUT_DIR, exist_ok=True)

for r in range(NUM_REDUCERS):
    incoming = [pair for out in mapper_outputs for pair in out[r]]
    result = reduce_phase(incoming)
    with open(os.path.join(OUT_DIR, f"part-{r}.txt"), "w") as f:
        for word, total in result:
            f.write(f"{word}\t{total}\n")
    print(f"--- part-{r}.txt ---")
    for word, total in result:
        print(word, total)

--- part-0.txt ---
damage 2
water 2
--- part-1.txt ---
car 3
fire 2
roof 2
theft 1
